# Callbacks, Caching, and Observability

Multi-step LangChain pipelines are hard to debug from print statements alone. **Callbacks** hook into every Runnable lifecycle event; **`RunnableConfig`** carries tags, metadata, and handlers through chains (**02**); **caching** avoids repeat LLM calls; and **LangSmith** (optional) records traces for inspection, eval, and monitoring.

This notebook covers custom **`BaseCallbackHandler`** implementations, **`RunnableConfig`**, **`with_config`**, **`InMemoryCache`** / **`set_llm_cache`**, **`astream_events`** for step-level visibility (**07**), LangSmith environment setup, tracing agents (**13**), and production observability patterns.

Prerequisites: **02. Runnable Protocol and LCEL**, **07. Streaming, Batch, and Async**, **11. RAG with LCEL**, **13. Agents with create_agent**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"
LANGCHAIN_API_KEY = "ls-placeholder-replace-with-your-langsmith-key"  # optional

import json
import os

import langchain_core

print("langchain-core:", langchain_core.__version__)

---

## 1. The Observability Stack

```
Request
   │
   ▼
RunnableConfig (tags, metadata, callbacks)
   │
   ├──► Callback handlers ──► logs / metrics / LangSmith
   ├──► LLM cache ──► skip duplicate model calls
   └──► Chain steps ──► astream_events for live UI
   │
   ▼
Response
```

| Tool | Layer | Best for |
|------|-------|----------|
| **`print` / logging** | Application | Quick local debug |
| **`BaseCallbackHandler`** | LangChain lifecycle | Step timing, custom metrics |
| **`RunnableConfig` tags/metadata** | Run labeling | Filter traces by route/user |
| **`astream_events`** | Streaming events | Progress UI, token streams |
| **`set_llm_cache`** | LLM responses | Cost/latency on repeated prompts |
| **LangSmith** | Hosted tracing | Team debugging, eval datasets |

Add observability **before** production — retrofitting traces after an outage is painful.

---

## 2. Setup — Chain and Agent

Reusable demo chain and a minimal agent for tracing examples:

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer backend engineering questions concisely."),
    ("human", "{question}"),
])

qa_chain = qa_prompt | llm | StrOutputParser()


@tool
def lookup_doc(topic: str) -> str:
    """Look up docs for alembic, jwt, or pytest."""
    docs = {
        "alembic": "Run alembic upgrade head to apply migrations.",
        "jwt": "JWT uses Authorization: Bearer header.",
        "pytest": "DB fixtures belong in conftest.py.",
    }
    for key, text in docs.items():
        if key in topic.lower():
            return text
    return "No doc found."


agent = create_agent(
    model=llm,
    tools=[lookup_doc],
    system_prompt="Use lookup_doc for factual questions.",
)

print("chain and agent ready")

---

## 3. RunnableConfig — Tags, Metadata, run_name

**`RunnableConfig`** propagates through every step in a chain invocation (**02**):

In [ ]:
from langchain_core.runnables import RunnableConfig

config = RunnableConfig(
    run_name="lesson15_qa",
    tags=["lesson-15", "qa-chain"],
    metadata={
        "notebook": "15-callbacks",
        "route": "/api/chat",
        "env": "dev",
    },
)

answer = qa_chain.invoke({"question": "What is Alembic?"}, config=config)
print(answer)

| Field | Purpose |
|-------|--------|
| **`run_name`** | Human-readable label in traces |
| **`tags`** | Filter/group runs (LangSmith, logs) |
| **`metadata`** | Arbitrary key-value (user id, tenant, version) |
| **`callbacks`** | Handlers attached to this run (**§4**) |
| **`max_concurrency`** | Batch parallelism (**07**) |
| **`configurable`** | Runtime-swappable fields (**16**) |

### 3.1 with_config — Default Config on a Chain

Bake observability defaults into the Runnable:

In [ ]:
tagged_chain = qa_chain.with_config({
    "run_name": "support_qa",
    "tags": ["support", "v1"],
    "metadata": {"product": "notes-api"},
})

print(tagged_chain.invoke({"question": "What is JWT?"}))

Per-request config **merges** with `with_config` defaults — override tags/metadata per invoke when needed.

---

## 4. Custom Callback Handlers

Subclass **`BaseCallbackHandler`** to react to lifecycle events:

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
from typing import Any


class StepLogger(BaseCallbackHandler):
    """Lightweight console tracer for chain and LLM steps."""

    def on_chain_start(
        self, serialized: dict[str, Any], inputs: dict[str, Any], **kwargs: Any
    ) -> None:
        name = kwargs.get("name") or serialized.get("name", "chain")
        print(f"  ▶ chain_start: {name}")

    def on_chain_end(self, outputs: dict[str, Any], **kwargs: Any) -> None:
        name = kwargs.get("name", "chain")
        preview = str(outputs)[:60].replace("\n", " ")
        print(f"  ◀ chain_end:   {name} → {preview}...")

    def on_chat_model_start(
        self, serialized: dict[str, Any], messages: list, **kwargs: Any
    ) -> None:
        print(f"  ▶ chat_model_start ({len(messages)} message batches)")

    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> None:
        text = response.generations[0][0].text if response.generations else ""
        print(f"  ◀ llm_end:       {text[:50]}...")


logger = StepLogger()
print("Running chain with StepLogger:")
qa_chain.invoke(
    {"question": "Name one pytest convention."},
    config=RunnableConfig(callbacks=[logger]),
)

### 4.1 Common Callback Events

| Event | Fires when |
|-------|------------|
| **`on_chain_start` / `on_chain_end`** | Any Runnable step begins / finishes |
| **`on_chat_model_start` / `on_llm_end`** | Chat model call starts / completes |
| **`on_chat_model_stream`** | Token chunk streamed |
| **`on_tool_start` / `on_tool_end`** | Tool execution (**12**, **13**) |
| **`on_retriever_start` / `on_retriever_end`** | Retriever search (**10**) |
| **`on_chain_error`** | Step raises an exception |

Use **`on_chain_error`** for alerting; **`on_llm_end`** for token usage logging (read from `response.llm_output`).

### 4.2 StdOutCallbackHandler

Built-in verbose printer for quick debugging:

In [ ]:
from langchain_core.callbacks import StdOutCallbackHandler

# Verbose — uncomment to see full stdout trace:
# qa_chain.invoke(
#     {"question": "What is FastAPI?"},
#     config=RunnableConfig(callbacks=[StdOutCallbackHandler()]),
# )
print("StdOutCallbackHandler available for verbose local debugging")

---

## 5. Timing Callback — Latency per Step

Measure where time is spent in RAG and agent pipelines:

In [ ]:
import time


class TimingCallback(BaseCallbackHandler):
    def __init__(self):
        self._starts: dict[str, float] = {}
        self.records: list[tuple[str, float]] = []

    def _key(self, kwargs: dict) -> str:
        return str(kwargs.get("run_id", id(self)))

    def on_chain_start(self, serialized, inputs, **kwargs):
        self._starts[self._key(kwargs)] = time.perf_counter()

    def on_chain_end(self, outputs, **kwargs):
        start = self._starts.pop(self._key(kwargs), None)
        if start is not None:
            name = kwargs.get("name", "chain")
            elapsed_ms = (time.perf_counter() - start) * 1000
            self.records.append((name, elapsed_ms))


timer = TimingCallback()
qa_chain.invoke({"question": "What is OAuth2?"}, config=RunnableConfig(callbacks=[timer]))

for name, ms in timer.records:
    print(f"{name:25s} {ms:7.1f} ms")

In production, emit **`records`** to Prometheus, Datadog, or OpenTelemetry instead of printing.

---

## 6. LLM Caching with InMemoryCache

**`set_llm_cache`** stores **exact** LLM responses keyed by prompt + model parameters. Repeated identical calls return cached output without an API round trip:

In [ ]:
from langchain_core.caches import InMemoryCache
from langchain_core.globals import get_llm_cache, set_llm_cache

set_llm_cache(InMemoryCache())
print("cache type:", type(get_llm_cache()).__name__)

In [ ]:
import time

question = {"question": "Define idempotency in one sentence."}

t0 = time.perf_counter()
first = qa_chain.invoke(question)
t1 = time.perf_counter()

second = qa_chain.invoke(question)
t2 = time.perf_counter()

print("first: ", first[:80], "...")
print(f"first took {1000*(t1-t0):.0f} ms | second took {1000*(t2-t1):.0f} ms")
print("answers match:", first == second)

### 6.1 Cache Scope and Limits

| Topic | Detail |
|-------|--------|
| **What is cached** | LLM/chat model outputs only |
| **What is not cached** | Embeddings, retriever results, tool side effects |
| **`InMemoryCache`** | Notebook/dev — lost on restart |
| **Production** | Redis/SQLite-backed caches via community integrations |
| **Staleness** | Cache ignores live data — do not cache time-sensitive answers |
| **Temperature > 0** | Same prompt may still miss cache if sampling differs |

Disable cache for production agents with side-effect tools. Reset with **`set_llm_cache(None)`**.

---

## 7. astream_events — Step-Level Visibility

**`astream_events`** emits structured events for every Runnable step (**07**). Useful for progress UIs and debugging multi-step pipelines:

In [ ]:
import asyncio


async def events_demo():
    print("astream_events (selected):")
    async for event in qa_chain.astream_events(
        {"question": "What is FastAPI?"},
        version="v2",
    ):
        kind = event.get("event")
        if kind == "on_chat_model_stream":
            chunk = event.get("data", {}).get("chunk")
            if chunk and getattr(chunk, "content", None):
                print(chunk.content, end="", flush=True)
        elif kind == "on_chain_end" and event.get("name") == "StrOutputParser":
            print("\n[parser finished]")
    print()


asyncio.run(events_demo())

### 7.1 Event Types for RAG and Agents

| Event | Typical source |
|-------|----------------|
| `on_retriever_start` / `on_retriever_end` | Vector search (**10**, **11**) |
| `on_chat_model_start` | Prompt sent to LLM |
| `on_chat_model_stream` | Answer tokens |
| `on_tool_start` / `on_tool_end` | Agent tool execution (**13**) |

Always pass **`version="v2"`** — the v1 schema is deprecated.

---

## 8. LangSmith Tracing

**[LangSmith](https://smith.langchain.com)** records nested traces for chains and agents. Enable via **environment variables** (recommended) or an explicit **`LangChainTracer`**:

In [ ]:
# Option A — environment variables (typical in CI and production)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
# os.environ["LANGCHAIN_PROJECT"] = "langchain-lesson-15"

print("LangSmith env vars (set before importing chains in production):")
print("  LANGCHAIN_TRACING_V2=true")
print("  LANGCHAIN_API_KEY=ls__...")
print("  LANGCHAIN_PROJECT=your-project-name")

In [ ]:
from langchain_core.tracers import LangChainTracer

# Option B — explicit tracer per request (no global env)
tracer = LangChainTracer(
    project_name="langchain-lesson-15",
)

traced_config = RunnableConfig(
    callbacks=[tracer],
    tags=["langsmith-demo"],
    metadata={"lesson": "15"},
)

# Uncomment when LANGCHAIN_API_KEY is set — sends trace to LangSmith:
# qa_chain.invoke({"question": "What is Alembic?"}, config=traced_config)
print("LangChainTracer configured (invoke commented until API key is set)")

In LangSmith you see: prompt inputs, model outputs, latency per step, tool calls, and errors — essential for debugging RAG recall vs generation failures (**11**).

### 8.1 @traceable — Non-LangChain Functions

Wrap plain Python functions in the same trace:

In [ ]:
from langsmith import traceable


@traceable(name="preprocess_question")
def preprocess_question(text: str) -> str:
    return text.strip().lower()


clean = preprocess_question("  What is JWT?  ")
print("preprocessed:", clean)

Use **`@traceable`** for preprocessing, database lookups, and business logic between LangChain steps.

---

## 9. Tracing Agents

Agents are LangGraph graphs — pass the same **`RunnableConfig`** to **`invoke`** / **`stream`**:

In [ ]:
agent_config = RunnableConfig(
    run_name="doc_lookup_agent",
    tags=["agent", "lesson-15"],
    metadata={"feature": "support-bot"},
    callbacks=[StepLogger()],
)

print("Running agent with StepLogger:")
agent_result = agent.invoke(
    {"messages": [HumanMessage(content="Look up jwt documentation.")]},
    config=agent_config,
)

final = agent_result["messages"][-1].content
print("final:", final[:120], "..." if len(str(final)) > 120 else "")

Agent traces include **multiple** model and tool nodes — LangSmith collapses them into a single nested run. Use tags to separate agent routes from simple chains.

---

## 10. Retriever and Tool Callbacks

Custom retrievers receive **`run_manager`** for callback propagation (**10**). Tools and retrievers fire **`on_tool_*`** / **`on_retriever_*`** events automatically when inside a traced chain:

In [ ]:
class RetrievalLogger(BaseCallbackHandler):
    def on_retriever_start(self, serialized, query, **kwargs):
        print(f"  ▶ retriever_start: {query!r}")

    def on_retriever_end(self, documents, **kwargs):
        print(f"  ◀ retriever_end:   {len(documents)} docs")

    def on_tool_start(self, serialized, input_str, **kwargs):
        name = serialized.get("name", "tool")
        print(f"  ▶ tool_start: {name} input={input_str!r}")

    def on_tool_end(self, output, **kwargs):
        preview = str(output)[:60].replace("\n", " ")
        print(f"  ◀ tool_end:   {preview}...")


print("RetrievalLogger ready — attach via RunnableConfig(callbacks=[RetrievalLogger()])")

Attach **`RetrievalLogger`** to RAG chains to confirm retrieval ran before generation — the first place to look when answers are wrong.

---

## 11. Error Callbacks

Capture failures without crashing the logging path:

In [ ]:
class ErrorLogger(BaseCallbackHandler):
    def on_chain_error(self, error: BaseException, **kwargs) -> None:
        name = kwargs.get("name", "chain")
        print(f"  ✖ chain_error: {name} → {type(error).__name__}: {error}")


broken_chain = qa_prompt | StrOutputParser()  # missing llm step

try:
    broken_chain.invoke(
        {"question": "test"},
        config=RunnableConfig(callbacks=[ErrorLogger()]),
    )
except Exception as exc:
    print(f"caught: {type(exc).__name__}")

Pair with **fallbacks** (**16. Fallbacks and Configurable Runnables**) — log the primary failure, then route to backup model.

---

## 12. What to Log in Production

| Log | Why | Avoid |
|-----|-----|-------|
| `run_name`, `tags`, `metadata` | Filter traces by tenant/route | Logging full prompts with PII |
| Per-step latency | Find slow retrieval vs LLM | Logging every token in prod |
| Tool names + args (redacted) | Audit agent behavior | Secrets in tool args |
| Retrieval doc ids | Debug RAG recall | Full doc bodies in logs |
| Error type + run_id | Incident correlation | Raw stack traces to clients |
| Token usage from `llm_output` | Cost tracking | — |

LangSmith can **mask** inputs/outputs — configure redaction for production projects.

---

## 13. Observability by Pipeline Type

| Pipeline | Primary hooks |
|----------|---------------|
| **Simple chain** | `RunnableConfig` + `StepLogger` |
| **RAG** (**11**) | `on_retriever_*` + `astream_events` |
| **Agent** (**13**) | LangSmith nested trace + `on_tool_*` |
| **Batch eval** (**07**) | `tags=["eval"]` + metadata per dataset row |
| **Cached dev loop** | `InMemoryCache` locally only |

Start with **`run_name` + tags** on every route, add LangSmith when the team needs shared traces.

---

## 14. Production Checklist

| Item | Action |
|------|--------|
| **Tracing** | `LANGCHAIN_TRACING_V2` in staging/prod |
| **Project separation** | Different `LANGCHAIN_PROJECT` per env |
| **Tags** | `env`, `route`, `model`, `tenant_id` |
| **Cache** | Dev only unless data is static |
| **PII** | Redact in LangSmith; avoid logging raw user text |
| **Alerts** | `on_chain_error` → monitoring system |
| **Cost** | Token usage metrics from callbacks |
| **Eval** | LangSmith datasets for regression (**09** eval notebook concepts) |

---

## 15. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| No tracing until prod incident | Blind debugging | Enable LangSmith in staging |
| Caching agent side effects | Stale wrong actions | `set_llm_cache(None)` for agents |
| Logging full RAG context | PII leakage | Log chunk ids only |
| Ignoring `on_retriever_end` | Miss retrieval bugs | Add retriever callback |
| `astream_events` v1 schema | Broken event parsing | `version="v2"` |
| Callback raises exception | Masks original error | Keep handlers minimal |
| Same tags for all routes | Unfilterable traces | Per-route `with_config` |
| Expecting cache on embeddings | Repeated embed cost | Separate embed cache/store |

---

## 16. Summary

**Observability** in LangChain layers **callbacks** (lifecycle hooks), **`RunnableConfig`** (tags/metadata), **caching** (LLM cost control), and **LangSmith** (hosted traces). Every Runnable honors the same config — chains, retrievers, and agents trace uniformly.

Key takeaways:

- **`BaseCallbackHandler`** implements step logging, timing, and error capture.
- **`RunnableConfig`** and **`with_config`** attach tags/metadata without changing chain logic.
- **`set_llm_cache(InMemoryCache())`** speeds dev; use carefully in production.
- **`astream_events(version="v2")`** exposes per-step events for UI and debug.
- **LangSmith** via env vars or **`LangChainTracer`** records nested agent traces.
- **`@traceable`** extends tracing to non-LangChain Python code.
- Log **ids and latencies**, not raw secrets or full document bodies.

Demonstrations built custom step/timing/error callbacks, configured RunnableConfig, demonstrated LLM caching, streamed chain events, sketched LangSmith setup, traced an agent run, and outlined production logging practices.

Next: **16. Fallbacks and Configurable Runnables** — resilient chains with backup models and runtime-swappable components.